# T7 It2: a candidate-level judge for answer-or-skip (no agreement)

**Loss targeted:** the decision after fine matching. Today's agreement-free rule (bar 8 on the 6 standard queries, else bar 11 on all 27) skips
right places with 4-10 inliers on official and still answers look-alike places with 8-11 inliers on fresh NUDT city photos (404, 505).

**Idea:** judge every candidate, not only the chosen one. Each photo has dozens of fine-matched wrong candidates (look-alike buildings), so the
five seen sets give ~9,600 labelled wrong and ~23,500 right candidates with >= 3 inliers, enough to learn what a chance match looks like from all the
signals available after fine matching (block `train/checks/decision_signals.py`, table from `It1_Signal_Table`). Model: monotone gradient
boosting (P(right) may only rise with inliers, DINO z, hull, coverage), block `train/checks/candidate_judge.py`.

**Decision:** pipeline's choice (most inliers + z, among candidates with >= 3 inliers); answer when P(right) >= tau or inliers >= 12; first on the
6 standard queries, then on all 27 (PC) or one extra query at a time up to 10 (Jetson, M = 5).

**Protocol:** leave one set out: the judge and its cut are fitted on the other four sets only (cut = smallest grid value with 0 wrong at it and at
every larger cut, then one grid step more), and the held-out set is replayed once. Baselines: the same replay with the inlier bars (no
agreement); Jetson: the submitted code's replay (bar 8, second pass, cap 10, M = 5).

In [1]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))   # one OpenMP thread per allowed core: more threads than cores slowed the judge ~100x
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.candidate_judge import (held_out_probabilities, FEATURES, FEATURES_CTX, add_context, add_features, baseline, decide, judge_probability,
                                          leave_one_set_out, paired_gain, summary)
RES = ROOT / "tests/Test_Iteration_7/results"
SETS = ["official", "nudt101", "nudt202", "nudt404", "nudt505"]
table = pd.concat([pd.read_csv(RES / f"signals/{s}.csv.gz") for s in SETS], ignore_index=True)
table = add_context(add_features(table)).reset_index(drop=True)
lab = table[(table.inliers >= 3)]
print(lab.assign(right=lab.err_sigma <= 4, wrong=lab.err_sigma > 8).groupby("data")[["right", "wrong"]].sum())
PROBS = held_out_probabilities(table, SETS, FEATURES_CTX)        # the five leave-one-set-out judges, trained once and reused below

          right  wrong
data                  
nudt101    5395   2963
nudt202    3622   2035
nudt404    3702   1827
nudt505    5312   2819
official   5475    744


## Baselines (no agreement)

In [2]:
base_pc = baseline(table, 8, 11, min_choice=3)
base_jet = baseline(table, 8, 8, jetson=True, m=5)
display(pd.concat({"PC: bar 8, then 27 queries @ 11": summary(base_pc), "Jetson today (M 5, 2nd pass @ 8, cap 10)": summary(base_jet)}))

score  wrong  answered  \
                                         data                                
PC: bar 8, then 27 queries @ 11          official  49.963      0     0.850   
                                         nudt101   54.840      0     0.917   
                                         nudt202   57.060      0     0.960   
                                         nudt404   54.240      2     0.940   
                                         nudt505   54.240      1     0.917   
Jetson today (M 5, 2nd pass @ 8, cap 10) official  50.280      0     0.855   
                                         nudt101   54.240      0     0.907   
                                         nudt202   55.260      0     0.930   
                                         nudt404   55.440      0     0.930   
                                         nudt505   54.680      1     0.927   

                                                   queries    n  n_answered  
                                         data                                
PC: bar 8, then 27 queries @ 11          official    9.491  379         322  
                                         nudt101     8.450  300         275  
                                         nudt202     7.470  200         192  
                                         nudt404     7.575  200         188  
                                         nudt505     8.380  300         275  
Jetson today (M 5, 2nd pass @ 8, cap 10) official    6.639  379         324  
                                         nudt101     6.433  300         272  
                                         nudt202     6.300  200         186  
                                         nudt404     6.310  200         186  
                                         nudt505     6.387  300         278

## Feature groups, leave one set out (PC)

In [3]:
GROUPS = {
    "inliers + z": ["li", "z"],
    "+ geometry (hull, cover9, spread)": ["li", "z", "hull", "cover9", "spread"],
    "+ fit (scale, angle, shift, resid)": ["li", "z", "hull", "cover9", "spread", "scale_dev", "angle", "shift_px", "resid"],
    "+ counts and confidence": ["li", "z", "hull", "cover9", "spread", "scale_dev", "angle", "shift_px", "resid", "ln22", "ratio", "tfrac", "conf_inl", "hc_ratio"],
    "all (+ rank, coarse, sharpness)": FEATURES,
    "all + query-local uniqueness": FEATURES_CTX,
}
abl = []
for name, feats in GROUPS.items():
    s, _ = leave_one_set_out(table, SETS, feats, override=12, min_choice=3)
    abl.append(s.assign(features=name))
abl = pd.concat(abl)
abl["cell"] = abl.score.map("{:.2f}".format) + " / " + abl.wrong.astype(str) + "w / t" + abl.tau.astype(str)
display(abl.pivot(index="features", columns="held", values="cell").loc[list(GROUPS)][SETS])
lg, _ = leave_one_set_out(table, SETS, FEATURES_CTX, kind="logit", override=12, min_choice=3)
print("logistic regression, all + uniqueness:"); display(lg)
abl.to_csv(RES / "it2_feature_groups_loso.csv", index=False)

held,official,nudt101,nudt202,nudt404,nudt505
features,,,,,
inliers + z,49.90 / 0w / t0.95,54.84 / 0w / t0.98,56.46 / 0w / t0.97,55.74 / 0w / t0.98,55.56 / 0w / t0.95
"+ geometry (hull, cover9, spread)",51.42 / 0w / t0.925,55.64 / 0w / t0.925,56.46 / 0w / t0.925,56.34 / 0w / t0.925,55.96 / 0w / t0.925
"+ fit (scale, angle, shift, resid)",51.42 / 0w / t0.9,55.44 / 0w / t0.925,56.46 / 0w / t0.925,56.70 / 0w / t0.9,56.16 / 0w / t0.925
+ counts and confidence,52.53 / 0w / t0.9,55.64 / 0w / t0.85,56.46 / 0w / t0.8,56.70 / 0w / t0.9,55.76 / 0w / t0.95
"all (+ rank, coarse, sharpness)",52.81 / 0w / t0.97,55.72 / 0w / t0.925,56.46 / 0w / t0.925,56.70 / 0w / t0.95,56.08 / 0w / t0.925
all + query-local uniqueness,54.36 / 0w / t0.95,55.72 / 0w / t0.97,56.46 / 0w / t0.97,56.70 / 0w / t0.97,55.96 / 0w / t0.97


logistic regression, all + uniqueness:


,held,tau,score,wrong,answered,queries
0,official,0.99,50.79,0,0.865,9.60
1,nudt101,0.99,54.72,0,0.913,8.59
2,nudt202,0.99,56.16,0,0.945,7.89
3,nudt404,0.99,55.74,0,0.935,8.10
4,nudt505,0.98,55.24,1,0.933,8.31


## The judge on the PC (all signals + uniqueness): held-out scores and paired gains

In [4]:
pc_sum, pc_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3)
pc_sum["vs PC baseline"] = pc_sum.held.map(paired_gain(base_pc, pc_rows, SETS))
pc_sum["baseline"] = pc_sum.held.map(lambda s: "%.2f / %dw" % (6 * base_pc[base_pc.data == s].pts.mean(), (base_pc[base_pc.data == s].pts < 0).sum()))
display(pc_sum)
nc_sum, nc_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=None, min_choice=3)
print("without the 12-inlier override:"); display(nc_sum)
cj_sum, cj_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3, choose="judge")
print("choosing the candidate by the judge (not by inliers + z):"); display(cj_sum)
pc_sum.to_csv(RES / "it2_judge_pc_loso.csv", index=False); pc_rows.to_csv(RES / "it2_judge_pc_loso_photos.csv", index=False)

,held,tau,score,wrong,answered,queries,vs PC baseline,baseline
0,official,0.95,54.36,0,0.926,8.49,"+4.40 [+2.88, +6.08]",49.96 / 0w
1,nudt101,0.97,55.72,0,0.930,8.31,"+0.88 [+0.08, +1.80]",54.84 / 0w
2,nudt202,0.97,56.46,0,0.950,7.78,"-0.60 [-1.50, +0.00]",57.06 / 0w
3,nudt404,0.97,56.70,0,0.955,7.47,"+2.46 [+0.48, +4.98]",54.24 / 2w
4,nudt505,0.97,55.96,0,0.937,8.03,"+1.72 [+0.60, +3.04]",54.24 / 1w


without the 12-inlier override:


,held,tau,score,wrong,answered,queries
0,official,0.95,54.36,0,0.926,8.49
1,nudt101,0.97,55.52,0,0.927,8.38
2,nudt202,0.97,55.86,0,0.935,8.20
3,nudt404,0.97,56.70,0,0.955,7.47
4,nudt505,0.97,55.36,0,0.927,8.24


choosing the candidate by the judge (not by inliers + z):


,held,tau,score,wrong,answered,queries
0,official,0.99,51.36,0,0.876,9.49
1,nudt101,0.99,55.64,0,0.930,8.31
2,nudt202,0.99,56.22,0,0.950,7.78
3,nudt404,0.99,56.58,0,0.945,7.89
4,nudt505,0.97,55.48,1,0.940,7.96


### Which photos change (PC, held-out decisions vs the baseline)

In [5]:
j = base_pc.set_index(["data", "photo"]).join(pc_rows.set_index(["data", "photo"]), rsuffix="_judge")
changed = j[j.pts != j.pts_judge][["stage", "pts", "inliers", "stage_judge", "pts_judge", "inliers_judge", "p_judge", "err_sigma_judge"]]
display(changed.round(2))
print(changed.groupby(level=0).apply(lambda g: pd.Series(dict(gained=(g.pts_judge > g.pts).sum(), lost=(g.pts_judge < g.pts).sum(),
                                                               wrong_removed=((g.pts < 0) & (g.pts_judge >= 0)).sum()))))

stage  pts  inliers stage_judge  pts_judge  inliers_judge  \
data     photo                                                             
official 7      skip    0      5.0        wide         10            5.0   
         26     skip    0      7.0        std6         10            7.0   
         50     skip    0      4.0        wide         10            4.0   
         57     skip    0      7.0        std6         10            7.0   
         66     skip    0     10.0        wide         10           10.0   
         72     skip    0      8.0        std6         10            6.0   
         76     skip    0     10.0        wide         10           10.0   
         99     skip    0     10.0        std6          6            6.0   
         110    skip    0      6.0        wide         10            6.0   
         122    skip    0      5.0        wide         10            5.0   
         126    skip    0      5.0        wide          6            5.0   
         144    skip    0      6.0        wide         10            6.0   
         154    skip    0      9.0        std6         10            5.0   
         172    skip    0      9.0        wide         10            9.0   
         179    skip    0      9.0        std6         10            5.0   
         182    skip    0      8.0        wide         10            8.0   
         201    skip    0      7.0        std6         10            6.0   
         210    skip    0      7.0        std6         10            6.0   
         222    skip    0      7.0        std6         10            7.0   
         234    skip    0      4.0        std6         10            4.0   
         241    skip    0      9.0        wide         10            9.0   
         287    skip    0      6.0        std6         10            6.0   
         325    skip    0      6.0        std6         10            6.0   
         326    skip    0      5.0        std6         10            5.0   
         329    skip    0      7.0        std6         10            7.0   
         338    skip    0      6.0        std6         10            6.0   
         345    skip    0      9.0        wide         10            9.0   
         365    skip    0      6.0        std6          6            6.0   
         376    skip    0      5.0        std6         10            5.0   
nudt101  37     skip    0      7.0        wide         10            7.0   
         254    skip    0      7.0        wide         10            7.0   
         255    skip    0      9.0        std6         10            7.0   
         449    skip    0      7.0        std6         10            5.0   
         451    skip    0     10.0        wide         10           10.0   
         454    wide    6     11.0        skip          0           11.0   
nudt202  227    std6   10     11.0        skip          0           11.0   
         256    std6   10      8.0        skip          0            8.0   
nudt404  76     skip    0      8.0        std6         10            7.0   
         188    skip    0      7.0        wide         10            7.0   
         211    std6  -20      9.0        wide         10           36.0   
         572    std6  -20      8.0        skip          0            8.0   
         1240   skip    0      9.0        std6          6            7.0   
         1315   skip    0      6.0        wide          6            6.0   
nudt505  482    skip    0      7.0        std6         10            6.0   
         602    skip    0     10.0        wide         10           10.0   
         668    skip    0     10.0        std6         10            7.0   
         889    wide  -20     11.0        skip          0           11.0   
         1208   skip    0      9.0        std6         10            6.0   
         1312   skip    0      6.0        std6         10            3.0   
         1325   skip    0     10.0        wide          6           10.0   
         2221   skip    0      8.0        wide         10      

          gained  lost  wrong_removed
data                                 
nudt101        5     1              0
nudt202        0     2              0
nudt404        6     0              2
nudt505        8     0              1
official      29     0              0


## The judge on the Jetson (M = 5, extra queries one at a time, cap 10)

In [6]:
jet_sum, jet_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3, jetson=True, m=5)
jet_sum["vs Jetson today"] = jet_sum.held.map(paired_gain(base_jet, jet_rows, SETS))
jet_sum["Jetson today"] = jet_sum.held.map(lambda s: "%.2f / %dw" % (6 * base_jet[base_jet.data == s].pts.mean(), (base_jet[base_jet.data == s].pts < 0).sum()))
jet_sum["modelled s / photo"] = (jet_sum.queries * 2.2).round(1)
display(jet_sum)
jet_sum.to_csv(RES / "it2_judge_jetson_loso.csv", index=False); jet_rows.to_csv(RES / "it2_judge_jetson_loso_photos.csv", index=False)

,held,tau,score,wrong,answered,queries,vs Jetson today,Jetson today,modelled s / photo
0,official,0.97,51.96,0,0.884,6.50,"+1.68 [+0.47, +2.94]",50.28 / 0w,14.3
1,nudt101,0.99,54.52,0,0.910,6.41,"+0.28 [-0.72, +1.28]",54.24 / 0w,14.1
2,nudt202,0.95,54.66,0,0.920,6.34,"-0.60 [-1.50, +0.00]",55.26 / 0w,13.9
3,nudt404,0.98,56.70,0,0.955,6.24,"+1.26 [+0.30, +2.46]",55.44 / 0w,13.7
4,nudt505,0.95,55.16,0,0.923,6.34,"+0.48 [-0.44, +1.60]",54.68 / 1w,13.9


## What the judge looks at (permutation importance on the held-out official set, judge trained on the four NUDT sets)

In [7]:
from sklearn.inspection import permutation_importance
from train.checks.candidate_judge import labelled
nudt = [s for s in SETS if s != "official"]
_, model = judge_probability(table, nudt, FEATURES_CTX)
off = table[(table.data == "official") & labelled(table)]
imp = permutation_importance(model, off[FEATURES_CTX].to_numpy(float), (off.err_sigma <= 4).to_numpy(), scoring="roc_auc", n_repeats=5, random_state=0)
display(pd.Series(imp.importances_mean, FEATURES_CTX).sort_values(ascending=False).round(4).to_frame("AUC drop"))

,AUC drop
zgap,0.0043
ligap,0.0032
hull,0.0018
li,0.0012
z,0.0003
spread,0.0001
rank,0.0001
sharp,0.0001
cover9,0.0001
ln22,0.0001


## Reading

- Inliers + z alone (what today's bars use) gains nothing on official under this protocol; the gain comes from the inlier geometry (hull,
  coverage), the counts (pairs, inlier ratio) and query-local uniqueness.
- 0 wrong answers on every held-out set, and the judge removes the wrong answers the plain bars make on fresh NUDT 404 (2) and 505 (1, photo 889,
  which every Test_Iteration_6 rule got wrong).
- NUDT 202 loses about 0.6: two right answers at 8 and 11 inliers get P 0.94, just under the cut.
- The cut picked on the training sets is 0.925-0.97, well above the break-even of a -20 / +10 game (~0.69): the margin is for safety, not tuned
  for score.